# Predicting Diabetes Risk from Behavioural and Health Indicators
### Traditional Machine Learning vs Deep Learning on the CDC BRFSS 2015 Survey

**Author:** Henriette Biziyaremye, African Leadership University  
**Module:** Machine Learning, Summative Assignment (Model Training and Evaluation)

**Problem.** Diabetes is frequently undiagnosed until complications appear, especially where routine blood-glucose screening is scarce. This notebook asks whether *low-cost, self-reported* indicators (blood pressure, cholesterol, BMI, general health, activity, income and similar) can flag people likely to have prediabetes or diabetes, so that scarce laboratory tests can be targeted.

**Task.** Binary classification: `Diabetes_binary` = 0 (no diabetes) vs 1 (prediabetes or diabetes).

**Notebook structure**
1. Setup and reproducibility
2. Data loading
3. Exploratory data analysis
4. Preprocessing and feature engineering
5. Evaluation and experiment-logging utilities
6. Traditional ML experiments (scikit-learn)
7. Deep learning experiments (TensorFlow/Keras + `tf.data`)
8. Experiment table
9. Final test-set evaluation of the best models
10. Error analysis
11. Saving models and artefacts
12. Conclusions

## 1. Setup and reproducibility
All randomness (Python, NumPy, TensorFlow, scikit-learn) is driven from a single `SEED`. Folders are created for figures, models, TensorBoard logs and result tables. The only package not preinstalled on Colab is `ucimlrepo` (used to download the dataset straight from UCI).

In [1]:
!pip install -q ucimlrepo tabulate tensorboard

In [ ]:
import os, json, time, random, zipfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.base import clone
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

warnings.filterwarnings('ignore')
SEED = 42

def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds()
for d in ['data', 'figures', 'models', 'logs', 'results']:
    os.makedirs(d, exist_ok=True)

sns.set_theme(style='whitegrid')
print('TensorFlow', tf.__version__, '| scikit-learn', sklearn.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Data loading
**Source:** *CDC Diabetes Health Indicators*, UCI Machine Learning Repository (ID 891), derived from the 2015 Behavioral Risk Factor Surveillance System (BRFSS), a US government telephone health survey. It contains 253,680 respondents and 21 features, well above the 20,000-row minimum, and is real survey data (not synthetic or a library toy dataset).

The raw file is saved and zipped in `data/` so it can be uploaded to the Google Drive folder. **It must not be committed to GitHub** (`data/` is in `.gitignore`).

In [ ]:
TARGET = 'Diabetes_binary'
RAW_CSV = 'data/cdc_diabetes_health_indicators_raw.csv'

try:
    from ucimlrepo import fetch_ucirepo
    cdc = fetch_ucirepo(id=891)
    df_raw = pd.concat([cdc.data.features, cdc.data.targets], axis=1)
except Exception as e:
    # Fallback: upload the Kaggle/UCI CSV manually to data/
    print('UCI download failed, using local CSV:', e)
    df_raw = pd.read_csv('data/diabetes_binary_health_indicators_BRFSS2015.csv')

df_raw.to_csv(RAW_CSV, index=False)
with zipfile.ZipFile('data/cdc_diabetes_raw.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(RAW_CSV, arcname=os.path.basename(RAW_CSV))

print('Raw shape:', df_raw.shape)
df_raw.head()

## 3. Exploratory data analysis
We check data types, missing values, duplicates, class balance, feature distributions and the univariate relationship of each feature with the target. These checks drive the preprocessing decisions in Section 4.

In [ ]:
print('Column dtypes:\n', df_raw.dtypes.value_counts(), '\n')
print('Total missing values:', int(df_raw.isna().sum().sum()))
print('Exact duplicate rows:', int(df_raw.duplicated().sum()))
df_raw.describe().T

In [ ]:
counts = df_raw[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No diabetes (0)', 'Prediabetes/diabetes (1)'], counts.values, color=['#4C72B0', '#DD8452'])
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}\n({v / counts.sum():.1%})', ha='center', va='bottom')
ax.set_ylabel('Respondents'); ax.set_title('Class distribution (raw data)')
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.savefig('figures/class_balance.png', dpi=150); plt.show()

**Interpretation.** Roughly 86% of respondents are in class 0, so a model that always predicts "no diabetes" would reach about 86% accuracy while being useless for screening. Accuracy is therefore a misleading headline metric here; the experiments are ranked by **PR-AUC** (average precision) and we also track recall, F1 and ROC-AUC.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, ['BMI', 'GenHlth', 'Age', 'Income', 'PhysHlth', 'HighBP']):
    kw = dict(bins=40) if col == 'BMI' else dict(discrete=True)
    sns.histplot(data=df_raw, x=col, hue=TARGET, stat='density', common_norm=False, ax=ax, **kw)
    ax.set_title(col)
plt.suptitle('Feature distributions by class (normalised within class)')
plt.tight_layout(); plt.savefig('figures/feature_distributions.png', dpi=150); plt.show()

In [ ]:
corr = df_raw.corr()[TARGET].drop(TARGET).sort_values()
corr.plot(kind='barh', figsize=(7, 8), color=['#C44E52' if v < 0 else '#4C72B0' for v in corr])
plt.title('Correlation of each feature with Diabetes_binary'); plt.xlabel('Pearson r')
plt.tight_layout(); plt.savefig('figures/target_correlation.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))
sns.boxplot(x=df_raw['BMI'], ax=ax); ax.set_title('BMI distribution (outlier check)')
plt.tight_layout(); plt.savefig('figures/bmi_boxplot.png', dpi=150); plt.show()
print(df_raw['BMI'].quantile([0.005, 0.5, 0.995]).round(1).to_dict(), '| max:', df_raw['BMI'].max())

**Interpretation.** General health (`GenHlth`), high blood pressure, BMI, walking difficulty, high cholesterol and age carry the strongest positive association with diabetes, while income, education and physical activity are protective, consistent with the clinical and social-determinants literature. BMI has a long right tail (values near 98 are implausible survey-entry errors or extreme cases), which motivates clipping. There are no missing values, but there are many exact duplicate rows.

## 4. Preprocessing and feature engineering

**4.1 Duplicates.** Because every feature is coarse (binary or a small ordinal scale), some identical rows may be genuinely different respondents. However, keeping them lets identical records fall into both training and test sets, which inflates test scores (a form of leakage). We remove them and acknowledge in the report that this slightly changes the class prior.

**4.2 Split.** A stratified 70/15/15 train/validation/test split. All tuning and model selection uses the validation set; the test set is touched **once**, in Section 9.

**4.3 Outliers.** BMI is clipped to the 0.5th–99.5th percentile *of the training set* (bounds are learned on training data only to avoid leakage).

**4.4 Engineered features** (each motivated by domain knowledge):
- `CardioRisk` = HighBP + HighChol + HeartDiseaseorAttack + Stroke: metabolic-syndrome style cumulative cardiovascular burden.
- `UnhealthyDays` = MentHlth + PhysHlth: overall recent health burden.
- `LifestyleScore` = PhysActivity + Fruits + Veggies − Smoker − HvyAlcoholConsump.
- `BMI_cat`: WHO BMI bands (underweight, normal, overweight, obese I–III), capturing non-linear risk.
- `AgeBMI` = Age × BMI: interaction, since excess weight matters more with age.
- `CareBarrier`: no health coverage or could not see a doctor due to cost (access to diagnosis).
- `PoorGenHlth`: self-rated health "fair" or "poor".

**4.5 Scaling.** Non-binary features are standardised (fit on training data). Binary features are left as 0/1. Tree models are scale-invariant, so the same matrices are used for all models for a fair comparison.

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True).astype(int)
print(f'Removed {len(df_raw) - len(df):,} duplicate rows -> {len(df):,} rows remain')
print('Class prior after de-duplication:', df[TARGET].value_counts(normalize=True).round(4).to_dict())

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df[TARGET], random_state=SEED)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name:5s}: {len(d):7,} rows | positive rate {d[TARGET].mean():.3f}')

In [ ]:
BASE_FEATURES = [c for c in df.columns if c != TARGET]
BMI_LOW, BMI_HIGH = train_df['BMI'].quantile([0.005, 0.995]).values

def engineer(d):
    d = d.copy()
    d['BMI'] = d['BMI'].clip(BMI_LOW, BMI_HIGH)
    d['CardioRisk'] = d['HighBP'] + d['HighChol'] + d['HeartDiseaseorAttack'] + d['Stroke']
    d['UnhealthyDays'] = d['MentHlth'] + d['PhysHlth']
    d['LifestyleScore'] = d['PhysActivity'] + d['Fruits'] + d['Veggies'] - d['Smoker'] - d['HvyAlcoholConsump']
    d['BMI_cat'] = pd.cut(d['BMI'], bins=[0, 18.5, 25, 30, 35, 40, np.inf], labels=False, right=False).astype(int)
    d['AgeBMI'] = d['Age'] * d['BMI']
    d['CareBarrier'] = ((d['AnyHealthcare'] == 0) | (d['NoDocbcCost'] == 1)).astype(int)
    d['PoorGenHlth'] = (d['GenHlth'] >= 4).astype(int)
    return d

train_e, val_e, test_e = engineer(train_df), engineer(val_df), engineer(test_df)
NEW_FEATURES = ['CardioRisk', 'UnhealthyDays', 'LifestyleScore', 'BMI_cat', 'AgeBMI', 'CareBarrier', 'PoorGenHlth']
ENG_FEATURES = BASE_FEATURES + NEW_FEATURES
y_train, y_val, y_test = (d[TARGET].values for d in (train_e, val_e, test_e))
print(f'BMI clipped to [{BMI_LOW:.0f}, {BMI_HIGH:.0f}]')

# Quick univariate check: are the engineered features informative?
train_e[NEW_FEATURES + ['BMI', 'GenHlth', 'HighBP']].corrwith(train_e[TARGET]).sort_values(ascending=False).round(3)

In [ ]:
def scale_sets(cols):
    binary = [c for c in cols if train_e[c].nunique() <= 2]
    to_scale = [c for c in cols if c not in binary]
    scaler = StandardScaler().fit(train_e[to_scale])
    def transform(d):
        out = d[cols].astype('float32').copy()
        out[to_scale] = scaler.transform(d[to_scale])
        return out.values.astype('float32')
    return {'train': transform(train_e), 'val': transform(val_e), 'test': transform(test_e),
            'scaler': scaler, 'scaled_cols': to_scale, 'columns': cols}

DATA = {'base': scale_sets(BASE_FEATURES), 'eng': scale_sets(ENG_FEATURES)}
for k, v in DATA.items():
    print(k, v['train'].shape, '| scaled:', v['scaled_cols'])

## 5. Evaluation and experiment-logging utilities
Every experiment goes through the same functions, so results are directly comparable:
- `metrics_at` computes accuracy, precision, recall, F1 (at threshold 0.5), ROC-AUC and PR-AUC.
- `best_f1_threshold` finds the decision threshold that maximises F1 on validation data (important under imbalance).
- `record` appends a row to the experiment log (saved to `results/experiment_log.csv`) **and** writes the same metrics to TensorBoard under `logs/<exp_id>/eval`, so every number in the report's table is traceable to a logged run.
- The train-vs-validation ROC-AUC gap (`auc_gap`) is logged as an overfitting indicator.

In [ ]:
RESULTS, MODELS, VAL_PROBS, HISTORIES, META = [], {}, {}, {}, {}
TR_IDX = np.random.RandomState(SEED).choice(len(y_train), size=min(50_000, len(y_train)), replace=False)

def take(X, idx):
    return {k: v[idx] for k, v in X.items()} if isinstance(X, dict) else X[idx]

def metrics_at(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {'accuracy': accuracy_score(y, yhat), 'precision': precision_score(y, yhat, zero_division=0),
            'recall': recall_score(y, yhat), 'f1': f1_score(y, yhat),
            'roc_auc': roc_auc_score(y, p), 'pr_auc': average_precision_score(y, p)}

def best_f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return float(thr[np.nanargmax(f1[:-1])])

def log_tb_scalars(exp_id, metrics):
    writer = tf.summary.create_file_writer(f'logs/{exp_id}/eval')
    with writer.as_default():
        for k, v in metrics.items():
            tf.summary.scalar(k, float(v), step=0)
    writer.flush()

def record(exp_id, paradigm, model_name, features, settings, p_val, train_auc, train_time, notes=''):
    global RESULTS
    m = metrics_at(y_val, p_val)
    thr = best_f1_threshold(y_val, p_val)
    m.update({'best_thr': thr, 'f1_at_best_thr': metrics_at(y_val, p_val, thr)['f1'],
              'train_roc_auc': train_auc, 'auc_gap': train_auc - m['roc_auc']})
    row = {'exp_id': exp_id, 'paradigm': paradigm, 'model': model_name, 'features': features,
           'split': '70/15/15 stratified; metrics on validation',
           'settings': json.dumps(settings, default=str),
           **{k: round(float(v), 4) for k, v in m.items()},
           'train_time_s': round(train_time, 1), 'notes': notes}
    RESULTS = [r for r in RESULTS if r['exp_id'] != exp_id] + [row]   # safe to re-run a cell
    log_tb_scalars(exp_id, m)
    pd.DataFrame(RESULTS).to_csv('results/experiment_log.csv', index=False)
    print(f"{exp_id}: PR-AUC={m['pr_auc']:.4f} ROC-AUC={m['roc_auc']:.4f} recall@0.5={m['recall']:.3f} "
          f"F1@0.5={m['f1']:.3f} F1@best={m['f1_at_best_thr']:.3f} (thr={thr:.2f}) gap={m['auc_gap']:.3f} [{train_time:.0f}s]")

## 6. Traditional machine learning experiments (scikit-learn)
The experiments are ordered so that each one changes **one thing** relative to an earlier run:

| ID | Change being tested |
|---|---|
| ML-1 | Logistic Regression baseline on the original 21 features |
| ML-2 | + balanced class weights (effect of imbalance handling) |
| ML-3 | + engineered features (effect of feature engineering) |
| ML-4 | L1-regularised LR (embedded feature selection) |
| ML-5 | Linear SVM (different linear loss, calibrated probabilities) |
| ML-6 | Random Forest, default settings (non-linear, high variance) |
| ML-7 | Random Forest, tuned by randomised search |
| ML-8 | Histogram Gradient Boosting, default |
| ML-9 | Histogram Gradient Boosting, tuned by randomised search |

Searches optimise **average precision** with 3-fold stratified CV on the training data only.

In [ ]:
def run_sklearn(exp_id, model, features, settings, notes=''):
    Xtr, Xva = DATA[features]['train'], DATA[features]['val']
    t0 = time.time(); model.fit(Xtr, y_train); tt = time.time() - t0
    p_val = model.predict_proba(Xva)[:, 1]
    train_auc = roc_auc_score(y_train[TR_IDX], model.predict_proba(Xtr[TR_IDX])[:, 1])
    MODELS[exp_id], VAL_PROBS[exp_id] = model, p_val
    META[exp_id] = {'paradigm': 'Traditional ML', 'features': features}
    record(exp_id, 'Traditional ML', type(model).__name__, features, settings, p_val, train_auc, tt, notes)
    return model

### ML-1 and ML-2: Logistic Regression baseline, with and without class weighting
Logistic Regression is interpretable and a strong baseline for tabular health data. ML-2 re-weights the loss so each class contributes equally; we expect recall to rise sharply and precision to fall, while ranking metrics (ROC-AUC, PR-AUC) change little, since weighting mainly shifts the decision boundary.

In [ ]:
run_sklearn('ML-1', LogisticRegression(max_iter=2000, random_state=SEED), 'base',
            {'C': 1.0, 'class_weight': None}, 'Baseline')
run_sklearn('ML-2', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED), 'base',
            {'C': 1.0, 'class_weight': 'balanced'}, 'Imbalance handling via class weights');

### ML-3: Logistic Regression + engineered features
Same as ML-2 but with the seven engineered features. Any gain here isolates the value of feature engineering for a linear model, which cannot learn interactions or BMI thresholds on its own.

In [ ]:
run_sklearn('ML-3', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED), 'eng',
            {'C': 1.0, 'class_weight': 'balanced'}, 'Adds engineered features');

### ML-4: L1-regularised Logistic Regression (embedded feature selection)
The L1 penalty drives uninformative coefficients to exactly zero. If performance holds with fewer features, the model is simpler and cheaper to deploy as a questionnaire.

In [ ]:
lr_l1 = run_sklearn('ML-4', LogisticRegression(penalty='l1', solver='liblinear', C=0.01, class_weight='balanced',
                                               max_iter=2000, random_state=SEED), 'eng',
                    {'penalty': 'l1', 'C': 0.01, 'class_weight': 'balanced'}, 'Embedded feature selection')
coefs = pd.Series(lr_l1.coef_[0], index=ENG_FEATURES)
print('Features dropped (coef = 0):', list(coefs[coefs == 0].index))
coefs.sort_values().plot(kind='barh', figsize=(7, 8)); plt.title('ML-4 L1-LR coefficients (standardised)')
plt.tight_layout(); plt.savefig('figures/ml4_l1_coefficients.png', dpi=150); plt.show()

### ML-5: Linear SVM
A hinge-loss linear classifier. `LinearSVC` has no probabilities, so it is wrapped in `CalibratedClassifierCV` (Platt scaling). A kernel SVM would be O(n²) on ~160k rows, which is computationally impractical; this trade-off is discussed in the report.

In [ ]:
svm = CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', dual=False, random_state=SEED),
                             cv=3, method='sigmoid')
run_sklearn('ML-5', svm, 'eng', {'C': 1.0, 'class_weight': 'balanced', 'calibration': 'sigmoid, cv=3'},
            'Linear SVM, calibrated');

### ML-6: Random Forest (default)
Fully grown trees can memorise the training data. We expect a very high training AUC and a large `auc_gap`, i.e. high variance.

In [ ]:
run_sklearn('ML-6', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED), 'eng',
            {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1}, 'Default RF (expect overfitting)');

### ML-7: Random Forest (tuned)
Randomised search over depth, leaf size, feature subsampling and class weighting, on a 60k-row training subsample to keep compute reasonable. The best configuration is then refit on the full training set. Shallower trees and larger leaves should trade a little bias for a large reduction in variance.

In [ ]:
SUB = np.random.RandomState(SEED).choice(len(y_train), size=min(60_000, len(y_train)), replace=False)
CV3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

rf_grid = {'n_estimators': [200, 400], 'max_depth': [8, 12, 16, None], 'min_samples_leaf': [1, 5, 20, 50],
           'max_features': ['sqrt', 0.5], 'class_weight': [None, 'balanced_subsample']}
rf_search = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1, random_state=SEED), rf_grid, n_iter=10,
                               scoring='average_precision', cv=CV3, random_state=SEED, verbose=1)
rf_search.fit(DATA['eng']['train'][SUB], y_train[SUB])
pd.DataFrame(rf_search.cv_results_).sort_values('rank_test_score').to_csv('results/rf_search.csv', index=False)
print('Best RF params:', rf_search.best_params_, '| CV AP:', round(rf_search.best_score_, 4))
run_sklearn('ML-7', RandomForestClassifier(**rf_search.best_params_, n_jobs=-1, random_state=SEED), 'eng',
            rf_search.best_params_, 'Tuned RF (RandomizedSearchCV, 10 iters, AP)');

### ML-8 and ML-9: Histogram Gradient Boosting (default, then tuned)
Boosting fits trees sequentially to the residual errors, reducing bias, and histogram binning makes it fast on large tabular data. Tuning covers learning rate, tree size, L2 regularisation, minimum leaf size and class weighting, with internal early stopping.

In [ ]:
run_sklearn('ML-8', HistGradientBoostingClassifier(random_state=SEED), 'eng',
            {'learning_rate': 0.1, 'max_iter': 100, 'max_leaf_nodes': 31}, 'Default HGB');

hgb_grid = {'learning_rate': [0.03, 0.05, 0.1], 'max_iter': [200, 400], 'max_leaf_nodes': [15, 31, 63],
            'l2_regularization': [0.0, 1.0, 10.0], 'min_samples_leaf': [20, 50, 100],
            'class_weight': [None, 'balanced']}
hgb_search = RandomizedSearchCV(HistGradientBoostingClassifier(early_stopping=True, random_state=SEED), hgb_grid,
                                n_iter=15, scoring='average_precision', cv=CV3, random_state=SEED, verbose=1)
hgb_search.fit(DATA['eng']['train'], y_train)
pd.DataFrame(hgb_search.cv_results_).sort_values('rank_test_score').to_csv('results/hgb_search.csv', index=False)
print('Best HGB params:', hgb_search.best_params_, '| CV AP:', round(hgb_search.best_score_, 4))
run_sklearn('ML-9', HistGradientBoostingClassifier(**hgb_search.best_params_, early_stopping=True,
                                                   random_state=SEED), 'eng',
            hgb_search.best_params_, 'Tuned HGB (RandomizedSearchCV, 15 iters, AP)');

In [ ]:
ml_res = pd.DataFrame(RESULTS).query("paradigm == 'Traditional ML'")
ml_res.plot(x='exp_id', y=['roc_auc', 'pr_auc', 'f1_at_best_thr'], kind='bar', figsize=(10, 4), rot=0)
plt.title('Traditional ML experiments (validation)'); plt.ylim(0, 1)
plt.tight_layout(); plt.savefig('figures/ml_comparison.png', dpi=150); plt.show()
ml_res[['exp_id', 'model', 'features', 'pr_auc', 'roc_auc', 'recall', 'f1', 'f1_at_best_thr', 'auc_gap']]

**Interpretation (write after running).** Compare: ML-1 vs ML-2 (class weights: recall should rise, ROC-AUC barely move); ML-2 vs ML-3 (does feature engineering help a linear model?); ML-6 vs ML-7 (the `auc_gap` should shrink sharply after tuning, a direct variance reduction); ML-8/ML-9 vs ML-3 (how much do non-linear interactions add over a well-engineered linear model?). Expect the gap between the best linear and best boosted model to be small, which suggests the signal in coarse self-reported survey features is largely additive.

## 7. Deep learning experiments (TensorFlow/Keras with `tf.data`)
All networks are fed through `tf.data` pipelines (shuffling, batching, prefetching). Each run uses:
- **EarlyStopping** on validation PR-AUC (patience 6, best weights restored),
- **ReduceLROnPlateau** on validation loss,
- a **TensorBoard** callback writing per-epoch training curves to `logs/<exp_id>`.

| ID | Change being tested |
|---|---|
| DL-1 | Sequential MLP 64-32, Adam 1e-3, no class weights (baseline) |
| DL-2 | + class weights |
| DL-3 | Deeper 256-128-64 + BatchNorm + Dropout 0.3 |
| DL-4 | DL-3 + L2 weight decay 1e-4 |
| DL-5 | DL-4 with learning rate 1e-2 (too high?) |
| DL-6 | DL-4 with learning rate 1e-4 (too low?) |
| DL-7 | DL-4 with batch size 2048 |
| DL-8 | Functional API wide-and-deep model with embeddings for ordinal features |

In [ ]:
CLASS_WEIGHTS = dict(enumerate(compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)))
print('Class weights:', {k: round(v, 3) for k, v in CLASS_WEIGHTS.items()})

def make_ds(X, y=None, batch_size=512, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(X if y is None else (X, y.astype('float32')))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(name='auc'), keras.metrics.AUC(curve='PR', name='pr_auc'),
                           keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')])
    return model

def build_mlp(input_dim, units=(64, 32), dropout=0.0, batchnorm=False, l2=0.0, lr=1e-3):
    reg = regularizers.l2(l2) if l2 > 0 else None
    model = keras.Sequential(name='mlp')
    model.add(keras.Input(shape=(input_dim,)))
    for u in units:
        model.add(layers.Dense(u, kernel_regularizer=reg))
        if batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid'))
    return compile_model(model, lr)

def plot_history(exp_id, h):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    for ax, m in zip(axes, ['loss', 'auc', 'pr_auc']):
        ax.plot(h[m], label='train'); ax.plot(h['val_' + m], label='validation')
        ax.set_title(f'{exp_id}: {m}'); ax.set_xlabel('epoch'); ax.legend()
    plt.tight_layout(); plt.savefig(f'figures/learning_curve_{exp_id}.png', dpi=150); plt.show()

def run_keras(exp_id, build_fn, features, settings, batch_size=512, epochs=40, class_weight=None, notes=''):
    set_seeds()
    model = build_fn()
    Xtr, Xva = DATA[features]['train'], DATA[features]['val']
    train_ds = make_ds(Xtr, y_train, batch_size, shuffle=True)
    val_ds = make_ds(Xva, y_val, batch_size)
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=6, restore_best_weights=True),
                 keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
                 keras.callbacks.TensorBoard(log_dir=f'logs/{exp_id}')]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weight,
                     callbacks=callbacks, verbose=2)
    tt = time.time() - t0
    p_val = model.predict(make_ds(Xva, batch_size=4096), verbose=0).ravel()
    p_tr = model.predict(make_ds(take(Xtr, TR_IDX), batch_size=4096), verbose=0).ravel()
    s = dict(settings, batch_size=batch_size, class_weight=class_weight is not None,
             epochs_run=len(hist.history['loss']), params=model.count_params())
    MODELS[exp_id], VAL_PROBS[exp_id], HISTORIES[exp_id] = model, p_val, hist.history
    META[exp_id] = {'paradigm': 'Deep Learning', 'features': features}
    record(exp_id, 'Deep Learning', model.name, features, s, p_val, roc_auc_score(y_train[TR_IDX], p_tr), tt, notes)
    plot_history(exp_id, hist.history)
    return model

N_IN = DATA['eng']['train'].shape[1]

### DL-1 and DL-2: baseline MLP, without and with class weights

In [ ]:
run_keras('DL-1', lambda: build_mlp(N_IN, (64, 32), lr=1e-3), 'eng',
          {'units': [64, 32], 'lr': 1e-3}, notes='Baseline MLP');

In [ ]:
run_keras('DL-2', lambda: build_mlp(N_IN, (64, 32), lr=1e-3), 'eng',
          {'units': [64, 32], 'lr': 1e-3}, class_weight=CLASS_WEIGHTS, notes='+ class weights');

### DL-3 and DL-4: more capacity, then regularisation
A wider, deeper network can model more interactions but risks overfitting; BatchNorm stabilises training and Dropout and L2 constrain variance. Watch the gap between training and validation loss in the learning curves.

In [ ]:
run_keras('DL-3', lambda: build_mlp(N_IN, (256, 128, 64), dropout=0.3, batchnorm=True, lr=1e-3), 'eng',
          {'units': [256, 128, 64], 'dropout': 0.3, 'batchnorm': True, 'lr': 1e-3},
          class_weight=CLASS_WEIGHTS, notes='Deeper + BN + dropout');

In [ ]:
run_keras('DL-4', lambda: build_mlp(N_IN, (256, 128, 64), dropout=0.3, batchnorm=True, l2=1e-4, lr=1e-3), 'eng',
          {'units': [256, 128, 64], 'dropout': 0.3, 'batchnorm': True, 'l2': 1e-4, 'lr': 1e-3},
          class_weight=CLASS_WEIGHTS, notes='+ L2 weight decay');

### DL-5 and DL-6: learning-rate sensitivity
A high learning rate (1e-2) should give noisy, unstable validation curves; a low one (1e-4) should converge slowly and may hit the epoch limit.

In [ ]:
run_keras('DL-5', lambda: build_mlp(N_IN, (256, 128, 64), dropout=0.3, batchnorm=True, l2=1e-4, lr=1e-2), 'eng',
          {'units': [256, 128, 64], 'dropout': 0.3, 'batchnorm': True, 'l2': 1e-4, 'lr': 1e-2},
          class_weight=CLASS_WEIGHTS, notes='High learning rate');

In [ ]:
run_keras('DL-6', lambda: build_mlp(N_IN, (256, 128, 64), dropout=0.3, batchnorm=True, l2=1e-4, lr=1e-4), 'eng',
          {'units': [256, 128, 64], 'dropout': 0.3, 'batchnorm': True, 'l2': 1e-4, 'lr': 1e-4},
          class_weight=CLASS_WEIGHTS, notes='Low learning rate');

### DL-7: large batch size
Larger batches give smoother gradient estimates and faster epochs, but fewer parameter updates per epoch. Without scaling the learning rate up, this usually slows convergence and can hurt generalisation.

In [ ]:
run_keras('DL-7', lambda: build_mlp(N_IN, (256, 128, 64), dropout=0.3, batchnorm=True, l2=1e-4, lr=1e-3), 'eng',
          {'units': [256, 128, 64], 'dropout': 0.3, 'batchnorm': True, 'l2': 1e-4, 'lr': 1e-3},
          batch_size=2048, class_weight=CLASS_WEIGHTS, notes='Batch size 2048');

### DL-8: Functional API wide-and-deep model with embeddings
Ordinal survey answers (general health, age band, education, income, BMI band) are given **learned embeddings** instead of being treated as plain numbers, so the network can learn non-monotonic effects. A linear "wide" path on the numeric inputs is added to the "deep" path's logit (Cheng et al., 2016), combining memorisation of simple linear effects with learned interactions. This requires the Functional API because the model has multiple named inputs.

In [ ]:
CAT_COLS = ['GenHlth', 'Age', 'Education', 'Income', 'BMI_cat']
CAT_DIMS = {c: int(max(train_e[c].max(), val_e[c].max(), test_e[c].max())) + 1 for c in CAT_COLS}
NUM_COLS = [c for c in ENG_FEATURES if c not in CAT_COLS]
wd_scale = [c for c in NUM_COLS if train_e[c].nunique() > 2]
wd_scaler = StandardScaler().fit(train_e[wd_scale])

def wd_inputs(d):
    num = d[NUM_COLS].astype('float32').copy()
    num[wd_scale] = wd_scaler.transform(d[wd_scale])
    x = {'num': num.values.astype('float32')}
    for c in CAT_COLS:
        x[c] = d[c].values.astype('int32').reshape(-1, 1)
    return x

DATA['wd'] = {'train': wd_inputs(train_e), 'val': wd_inputs(val_e), 'test': wd_inputs(test_e),
              'scaler': wd_scaler, 'scaled_cols': wd_scale, 'columns': NUM_COLS + CAT_COLS}

def build_wide_deep(num_dim, cat_dims, emb_dim=4, units=(128, 64), dropout=0.3, l2=1e-4, lr=1e-3):
    reg = regularizers.l2(l2)
    inputs = {'num': keras.Input(shape=(num_dim,), name='num')}
    embeddings = []
    for c, n in cat_dims.items():
        inputs[c] = keras.Input(shape=(1,), name=c, dtype='int32')
        embeddings.append(layers.Flatten()(layers.Embedding(n, emb_dim, name=f'emb_{c}')(inputs[c])))
    deep = layers.Concatenate()([inputs['num']] + embeddings)
    for u in units:
        deep = layers.Dense(u, kernel_regularizer=reg)(deep)
        deep = layers.BatchNormalization()(deep)
        deep = layers.Activation('relu')(deep)
        deep = layers.Dropout(dropout)(deep)
    logit = layers.Add()([layers.Dense(1, name='deep_logit')(deep), layers.Dense(1, name='wide_logit')(inputs['num'])])
    out = layers.Activation('sigmoid', name='prob')(logit)
    return compile_model(keras.Model(inputs=inputs, outputs=out, name='wide_and_deep'), lr)

build_wide_deep(len(NUM_COLS), CAT_DIMS).summary()
run_keras('DL-8', lambda: build_wide_deep(len(NUM_COLS), CAT_DIMS), 'wd',
          {'api': 'functional', 'emb_dim': 4, 'units': [128, 64], 'dropout': 0.3, 'l2': 1e-4, 'lr': 1e-3},
          class_weight=CLASS_WEIGHTS, notes='Wide & deep with embeddings');

**Interpretation (write after running).** Use the learning-curve figures: if training loss keeps falling while validation loss flattens or rises, the model is overfitting (variance); if both plateau at a similar, high value, it is limited by bias or by the information in the features. Compare DL-3 vs DL-4 (does L2 close the train/validation gap?), DL-5 (instability: jagged validation curves) and DL-6 (slow convergence), and DL-8 vs DL-4 (do embeddings help?).

To view all logged runs interactively, run the TensorBoard cell below.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 8. Experiment table
The complete, ordered log of all experiments. Every row was produced by `record()` and is also stored as TensorBoard scalars in `logs/<exp_id>/eval`, so the table in the report is traceable to logged runs. All metrics are on the **validation** split; the test set is reserved for Section 9.

In [ ]:
COLS = ['exp_id', 'paradigm', 'model', 'features', 'settings', 'split', 'accuracy', 'precision', 'recall', 'f1',
        'roc_auc', 'pr_auc', 'best_thr', 'f1_at_best_thr', 'train_roc_auc', 'auc_gap', 'train_time_s', 'notes']
exp_table = pd.DataFrame(RESULTS)[COLS]
exp_table.to_csv('results/experiment_table.csv', index=False)
with open('results/experiment_table.md', 'w') as f:
    f.write(exp_table.to_markdown(index=False))
pd.set_option('display.max_colwidth', 80)
exp_table

## 9. Final evaluation on the held-out test set
The best traditional and best deep-learning models are chosen by **validation PR-AUC**. Their decision thresholds are tuned on validation (max F1) and then applied unchanged to the test set, so the test numbers are an unbiased estimate.

In [ ]:
res = pd.DataFrame(RESULTS)
best_ml_id = res.query("paradigm == 'Traditional ML'").sort_values('pr_auc', ascending=False).iloc[0]['exp_id']
best_dl_id = res.query("paradigm == 'Deep Learning'").sort_values('pr_auc', ascending=False).iloc[0]['exp_id']
baseline_id = 'ML-1'
print('Best traditional:', best_ml_id, '| Best deep learning:', best_dl_id)

def predict(exp_id, split):
    X = DATA[META[exp_id]['features']][split]
    m = MODELS[exp_id]
    if META[exp_id]['paradigm'] == 'Deep Learning':
        return m.predict(make_ds(X, batch_size=4096), verbose=0).ravel()
    return m.predict_proba(X)[:, 1]

THR, P_TEST, rows = {}, {}, []
for eid in [baseline_id, best_ml_id, best_dl_id]:
    THR[eid] = best_f1_threshold(y_val, VAL_PROBS[eid])
    P_TEST[eid] = predict(eid, 'test')
    for label, t in [('0.50', 0.5), (f'tuned {THR[eid]:.2f}', THR[eid])]:
        rows.append({'exp_id': eid, 'threshold': label, **{k: round(v, 4) for k, v in metrics_at(y_test, P_TEST[eid], t).items()}})
test_table = pd.DataFrame(rows)
test_table.to_csv('results/test_results.csv', index=False)
test_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, eid in zip(axes, [best_ml_id, best_dl_id]):
    yhat = (P_TEST[eid] >= THR[eid]).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test, yhat), display_labels=['No diabetes', 'Diabetes/pre']).plot(
        ax=ax, cmap='Blues', values_format=',')
    ax.set_title(f'{eid} (test, threshold {THR[eid]:.2f})')
plt.tight_layout(); plt.savefig('figures/confusion_matrices_test.png', dpi=150); plt.show()

for eid in [best_ml_id, best_dl_id]:
    print(f'--- {eid} ---')
    print(classification_report(y_test, (P_TEST[eid] >= THR[eid]).astype(int), target_names=['No diabetes', 'Diabetes/pre'], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for eid in [baseline_id, best_ml_id, best_dl_id]:
    fpr, tpr, _ = roc_curve(y_test, P_TEST[eid])
    axes[0].plot(fpr, tpr, label=f'{eid} (AUC={roc_auc_score(y_test, P_TEST[eid]):.3f})')
    prec, rec, _ = precision_recall_curve(y_test, P_TEST[eid])
    axes[1].plot(rec, prec, label=f'{eid} (AP={average_precision_score(y_test, P_TEST[eid]):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1); axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curves (test)')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'No-skill ({y_test.mean():.3f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-recall curves (test)')
for ax in axes: ax.legend()
plt.tight_layout(); plt.savefig('figures/roc_pr_test.png', dpi=150); plt.show()

### Learning curve for the best traditional model
Training-set size is varied (3-fold CV on a 60k subsample) to diagnose bias vs variance: converging curves at a modest score indicate a bias/information ceiling; a persistent gap indicates variance that more data could reduce.

In [ ]:
sizes, tr_sc, va_sc = learning_curve(clone(MODELS[best_ml_id]), DATA[META[best_ml_id]['features']]['train'][SUB],
                                     y_train[SUB], train_sizes=np.linspace(0.1, 1.0, 6), cv=CV3,
                                     scoring='roc_auc', n_jobs=-1)
plt.figure(figsize=(7, 4.5))
for sc, lab in [(tr_sc, 'training'), (va_sc, 'cross-validation')]:
    plt.plot(sizes, sc.mean(1), 'o-', label=lab)
    plt.fill_between(sizes, sc.mean(1) - sc.std(1), sc.mean(1) + sc.std(1), alpha=0.2)
plt.xlabel('Training examples'); plt.ylabel('ROC-AUC'); plt.title(f'Learning curve: {best_ml_id}'); plt.legend()
plt.tight_layout(); plt.savefig(f'figures/learning_curve_{best_ml_id}.png', dpi=150); plt.show()

## 10. Error analysis
We examine *who* the best model gets wrong. False negatives (missed diabetes cases) are the costly error in a screening setting. We compare average profiles of TP/FN/FP/TN groups, compute the miss rate by subgroup (age, self-rated health, BMI band, income, sex), and use permutation importance to see which features the model relies on.

In [ ]:
eid = best_ml_id
err = test_e.copy()
err['pred'] = (P_TEST[eid] >= THR[eid]).astype(int)
err['outcome'] = np.select([(err[TARGET] == 1) & (err.pred == 1), (err[TARGET] == 1) & (err.pred == 0),
                            (err[TARGET] == 0) & (err.pred == 1)], ['TP', 'FN', 'FP'], 'TN')
profile_cols = ['BMI', 'Age', 'GenHlth', 'HighBP', 'HighChol', 'CardioRisk', 'PhysHlth', 'DiffWalk', 'Income']
profile = err.groupby('outcome')[profile_cols].mean().round(2).loc[['TP', 'FN', 'FP', 'TN']]
profile.to_csv('results/error_profile.csv')
profile

In [ ]:
pos = err[err[TARGET] == 1]
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8), sharey=True)
for ax, col in zip(axes, ['Age', 'GenHlth', 'BMI_cat', 'Income', 'Sex']):
    miss = 1 - pos.groupby(col)['pred'].mean()
    miss.plot(kind='bar', ax=ax, color='#C44E52', rot=0); ax.set_title(f'Miss rate by {col}')
axes[0].set_ylabel('False-negative rate among true positives')
plt.tight_layout(); plt.savefig('figures/miss_rate_by_subgroup.png', dpi=150); plt.show()

In [ ]:
VI = np.random.RandomState(SEED).choice(len(y_val), size=min(20_000, len(y_val)), replace=False)
pi = permutation_importance(MODELS[best_ml_id], DATA[META[best_ml_id]['features']]['val'][VI], y_val[VI],
                            scoring='average_precision', n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=DATA[META[best_ml_id]['features']]['columns']).sort_values()
imp.plot(kind='barh', figsize=(7, 8)); plt.title(f'Permutation importance ({best_ml_id}, drop in AP)')
plt.tight_layout(); plt.savefig('figures/permutation_importance.png', dpi=150); plt.show()

**Interpretation (write after running).** Typical findings to check for: false negatives tend to look like true negatives (younger, lower BMI, good self-rated health, no high blood pressure), i.e. diabetes cases *without* the classic risk profile are the hardest to detect from survey answers alone; false positives look like true positives (older, obese, hypertensive people who do not report diabetes), some of whom may actually be undiagnosed, a label-noise issue inherent to self-reported outcomes. If the miss rate is much higher for younger age bands or certain income groups, discuss fairness implications for screening.

## 11. Saving models and artefacts
The best traditional model is saved with `joblib` together with everything needed to reproduce its preprocessing (scaler, feature order, BMI clip bounds, tuned threshold). The best deep-learning model is saved in both `.keras` and `.h5` format. Models and TensorBoard logs are zipped for the Google Drive folder.

In [ ]:
f = META[best_ml_id]['features']
joblib.dump({'model': MODELS[best_ml_id], 'scaler': DATA[f]['scaler'], 'scaled_cols': DATA[f]['scaled_cols'],
             'features': DATA[f]['columns'], 'bmi_clip': (float(BMI_LOW), float(BMI_HIGH)),
             'threshold': THR[best_ml_id], 'exp_id': best_ml_id}, 'models/best_traditional_model.joblib')

best_dl = MODELS[best_dl_id]
best_dl.save('models/best_dl_model.keras')
try:
    best_dl.save('models/best_dl_model.h5')
except Exception as e:
    print('H5 export skipped:', e)
f = META[best_dl_id]['features']
joblib.dump({'scaler': DATA[f]['scaler'], 'scaled_cols': DATA[f]['scaled_cols'], 'features': DATA[f]['columns'],
             'bmi_clip': (float(BMI_LOW), float(BMI_HIGH)), 'threshold': THR[best_dl_id], 'exp_id': best_dl_id},
            'models/best_dl_preprocessing.joblib')

shutil.make_archive('models_bundle', 'zip', 'models')
shutil.make_archive('tensorboard_logs', 'zip', 'logs')
print(os.listdir('models'))

**Optional (Colab):** copy the zipped dataset, models and logs straight to your Google Drive folder.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DEST = '/content/drive/MyDrive/ML_Summative_Diabetes'
# os.makedirs(DEST, exist_ok=True)
# for p in ['data/cdc_diabetes_raw.zip', 'models_bundle.zip', 'tensorboard_logs.zip']:
#     shutil.copy(p, DEST)

## 12. Conclusions
*(Complete after running, in your own words.)* Summarise: which paradigm performed best on test PR-AUC/ROC-AUC and by how much; whether the extra complexity of deep learning was justified for this tabular, coarse-grained data (accuracy vs interpretability vs compute trade-off); what class weighting, feature engineering and regularisation each contributed; the main error patterns; and the key dataset limitations (self-reported labels, US-only 2015 population, prediabetes merged with diabetes, coarse ordinal features). Close with concrete next steps, e.g. validating on African data such as Rwanda's STEPS survey, cost-sensitive threshold selection with clinicians, and probability calibration.